# imports

In [1]:
import os 
import random
import numpy as np
import pandas as pd
import scipy
import sklearn
os.getcwd()

'c:\\Users\\juvad3723\\.1\\--Projects\\Local\\old_harry_paper\\data\\current_speed_pred'

In [2]:
data = None

for n in range(11):
    s = f'_{n}'
    csv_path = f'export{s}.csv'
    if n == 0:
        csv_path = 'export.csv'

    df = pd.read_csv(csv_path, encoding='ISO-8859-1')
    df = df[['date',"Temp. de l'eau", 'Salinité', 'Pression', 'Densité',
             'Période de vague','Hauteur de vague moy.', 'Hauteur de vague max.', 'Vitesse du vent',
            'Rafales', 'Direction du vent', 'Temp. air', 'Humidité','Courant-Direction(0m)','Courant - Vitesse (0m)'
            # 'Fluorescence','CDOM','Eau - Rétrodiffusion','Courant - Vitesse (6m)', 'Courant-Direction(6m)'
            ]]
    
    if data is None:
        data = df.copy()
    else:
        data = pd.concat([data, df], ignore_index=True)

print(len(data))
print(data.isna().sum())
data.head()


60991
date                       0
Temp. de l'eau             3
Salinité                   3
Pression                   1
Densité                    3
Période de vague          16
Hauteur de vague moy.     16
Hauteur de vague max.     15
Vitesse du vent            4
Rafales                    4
Direction du vent          4
Temp. air                  1
Humidité                   1
Courant-Direction(0m)      1
Courant - Vitesse (0m)     1
dtype: int64


,date,Temp. de l'eau,Salinité,Pression,Densité,Période de vague,Hauteur de vague moy.,Hauteur de vague max.,Vitesse du vent,Rafales,Direction du vent,Temp. air,Humidité,Courant-Direction(0m),Courant - Vitesse (0m)
0,2015-06-19T14:03Z[UTC],7.1,31.1,101.000000,1024.2,2.5,0.7,1.6,28.0,32.0,184.0,8.4,83.0,65.0,0.0
1,2015-06-19T14:18Z[UTC],7.1,31.1,101.000000,1024.2,2.6,0.8,1.7,29.0,35.0,156.0,8.4,84.0,30.0,0.0
2,2015-06-19T14:33Z[UTC],7.1,31.1,101.000000,1024.2,2.6,0.8,1.8,26.0,33.0,189.0,8.3,85.0,44.0,0.0
3,2015-06-19T14:48Z[UTC],7.1,31.1,101.020004,1024.2,2.5,0.7,1.4,24.0,29.0,183.0,8.2,85.0,43.0,0.0
4,2015-06-19T15:03Z[UTC],7.1,31.1,101.020004,1024.2,2.6,0.7,1.5,22.0,27.0,183.0,8.1,87.0,47.0,0.0


In [3]:
df = data.dropna(subset=['Courant - Vitesse (0m)'])
print(len(df))
print(df.isna().sum())
df.head()

60990
date                       0
Temp. de l'eau             2
Salinité                   2
Pression                   0
Densité                    2
Période de vague          15
Hauteur de vague moy.     15
Hauteur de vague max.     15
Vitesse du vent            3
Rafales                    3
Direction du vent          3
Temp. air                  0
Humidité                   0
Courant-Direction(0m)      0
Courant - Vitesse (0m)     0
dtype: int64


,date,Temp. de l'eau,Salinité,Pression,Densité,Période de vague,Hauteur de vague moy.,Hauteur de vague max.,Vitesse du vent,Rafales,Direction du vent,Temp. air,Humidité,Courant-Direction(0m),Courant - Vitesse (0m)
0,2015-06-19T14:03Z[UTC],7.1,31.1,101.000000,1024.2,2.5,0.7,1.6,28.0,32.0,184.0,8.4,83.0,65.0,0.0
1,2015-06-19T14:18Z[UTC],7.1,31.1,101.000000,1024.2,2.6,0.8,1.7,29.0,35.0,156.0,8.4,84.0,30.0,0.0
2,2015-06-19T14:33Z[UTC],7.1,31.1,101.000000,1024.2,2.6,0.8,1.8,26.0,33.0,189.0,8.3,85.0,44.0,0.0
3,2015-06-19T14:48Z[UTC],7.1,31.1,101.020004,1024.2,2.5,0.7,1.4,24.0,29.0,183.0,8.2,85.0,43.0,0.0
4,2015-06-19T15:03Z[UTC],7.1,31.1,101.020004,1024.2,2.6,0.7,1.5,22.0,27.0,183.0,8.1,87.0,47.0,0.0


In [4]:
df = df.fillna(0)
df.reset_index(drop=True, inplace=True)
df_rf = df.drop(columns='date')
df_rf.rename(columns={'Courant - Vitesse (0m)':'current_speed_m_s'}, inplace=True, errors='ignore')
print(df_rf.isna().sum())

Temp. de l'eau           0
Salinité                 0
Pression                 0
Densité                  0
Période de vague         0
Hauteur de vague moy.    0
Hauteur de vague max.    0
Vitesse du vent          0
Rafales                  0
Direction du vent        0
Temp. air                0
Humidité                 0
Courant-Direction(0m)    0
current_speed_m_s        0
dtype: int64


# RF model (unused)

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

# Features
X = df_rf.drop(columns=['current_speed_m_s'])

# dependent variable
y = df_rf['current_speed_m_s']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=24)

# Define model
model = RandomForestRegressor(n_estimators=200, max_depth=20, min_samples_leaf=2, random_state=24)

# Train
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Evaluate
mse = mean_squared_error(y_test, y_pred)
print("Mean Squared Error:", mse)




Mean Squared Error: 0.002195272060483372


random search

In [6]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import randint

# Features and target
X = df_rf.drop(columns=['current_speed_m_s'])
y = df_rf['current_speed_m_s']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=24
)

# Base model
rf_model = RandomForestRegressor(random_state=24, n_jobs=-1)

# Define hyperparameter distributions for random search
param_dist = {
    'n_estimators': randint(100, 1000),        # number of trees
    'max_depth': randint(5, 30),               # maximum tree depth
    'min_samples_split': randint(2, 10),       # min samples to split a node
    'min_samples_leaf': randint(1, 5),         # min samples per leaf
    'max_features': ['auto', 'sqrt', 'log2', None]  # features per split
}

# Randomized search with 50 iterations and 3-fold CV
random_search = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=param_dist,
    n_iter=50,         # number of random combinations
    scoring='r2',      # maximize R²
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

# Run randomized search
random_search.fit(X_train, y_train)

# Best parameters
print("Best parameters:", random_search.best_params_)

# Evaluate on test set
best_rf = random_search.best_estimator_
y_pred = best_rf.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = best_rf.score(X_test, y_test)

print(f"Test MSE: {mse:.4f}")
print(f"Test R²: {r2:.4f}")


Fitting 3 folds for each of 50 candidates, totalling 150 fits


c:\Users\juvad3723\AppData\Local\miniforge3\envs\geo-env\Lib\site-packages\sklearn\model_selection\_validation.py:516: FitFailedWarning: 
51 fits failed out of a total of 150.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
25 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\juvad3723\AppData\Local\miniforge3\envs\geo-env\Lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\juvad3723\AppData\Local\miniforge3\envs\geo-env\Lib\site-packages\sklearn\base.py", line 1358, in wrapper
    estimator._validate_params()
  File "c:\Users\juvad3723\AppData\Local\miniforge3\envs\geo-env\Lib\site-packages\sklear

Best parameters: {'max_depth': 24, 'max_features': None, 'min_samples_leaf': 1, 'min_samples_split': 4, 'n_estimators': 584}
Test MSE: 0.0021
Test R²: 0.7016


In [7]:
print('mean',y.mean().round(4))
print('median',y.median().round(4))
print('std',y.std().round(4))
print('min',y.min().round(4))
print('max',y.max().round(4))

mean 0.2593
median 0.3
std 0.0839
min 0.0
max 0.9


# xgboost

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
import xgboost as xgb

# Features (all columns except target)
X = df_rf.drop(columns=['current_speed_m_s'])
y = df_rf['current_speed_m_s']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=24
)

# Define model
model = xgb.XGBRegressor(
    n_estimators=1700,       # number of boosting rounds
    learning_rate=0.02,     # smaller = slower but better generalization
    max_depth=12,            # depth of trees
    subsample=0.8,          # use 80% of rows per tree
    colsample_bytree=0.8,   # use 80% of features per tree
    tree_method="hist",     # fast histogram-based method (CPU)
    random_state=24,
    n_jobs=-1               # use all CPU cores
)

# If you have a strong NVIDIA GPU, switch tree_method to "gpu_hist"
# model = xgb.XGBRegressor(tree_method="gpu_hist", predictor="gpu_predictor")

# Train
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Evaluate
rmse = root_mean_squared_error(y_test, y_pred)
print("rmse:", rmse)
r2 = r2_score(y_test, y_pred)
print("R²:", r2)

rmse: 0.04170784524445444
R²: 0.7524108331944439


In [17]:
import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.metrics import root_mean_squared_error
from scipy.stats import uniform, randint

# Features (all columns except target)
X = df_rf.drop(columns=['current_speed_m_s'])
y = df_rf['current_speed_m_s']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=30
)

# Base model
model = xgb.XGBRegressor(
    tree_method="hist",  # CPU training
    random_state=24,
    n_jobs=-1
)

# Define distributions for random search
param_dist = {
    'n_estimators': randint(1000, 2000),
    'learning_rate': uniform(0.01, 0.1),        # range 0.01–0.11
    'max_depth': randint(6, 13),                # 6–12
    'subsample': uniform(0.6, 0.3),             # 0.6–0.9
    'colsample_bytree': uniform(0.6, 0.3),      # 0.6–0.9
    'min_child_weight': randint(1, 6)           # 1–5
}

# Randomized search with 50 iterations and 3-fold CV
random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_dist,
    n_iter=250,           # number of random combinations to try
    scoring='r2',
    cv=3,
    verbose=2,
    random_state=24,
    n_jobs=-1
)

# Run the randomized search
random_search.fit(X_train, y_train)

# Best parameters
print("Best parameters:", random_search.best_params_)

# Evaluate on test set
best_model = random_search.best_estimator_
y_pred = best_model.predict(X_test)
rmse = root_mean_squared_error(y_test, y_pred)
r2 = best_model.score(X_test, y_test)

print(f"Test RMSE: {rmse:.4f}")
print(f"Test R²: {r2:.4f}")

Fitting 3 folds for each of 250 candidates, totalling 750 fits
Best parameters: {'colsample_bytree': np.float64(0.7937181442322921), 'learning_rate': np.float64(0.012610979637111256), 'max_depth': 12, 'min_child_weight': 5, 'n_estimators': 1960, 'subsample': np.float64(0.6835260930401149)}
Test RMSE: 0.0414
Test R²: 0.7549


In [44]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
import xgboost as xgb

# Features (all columns except target)
X = df_rf.drop(columns=['current_speed_m_s'])
y = df_rf['current_speed_m_s']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=24
)

# Define model
model = xgb.XGBRegressor(
    n_estimators=2000,     
    learning_rate=0.02,    
    max_depth=13,          
    subsample=0.7,          # use % of rows per tree
    colsample_bytree=0.8,   # use % of features per tree
    tree_method="hist",  
    random_state=22,
    min_child_weight=5,
    n_jobs=-1               # use all CPU cores
)

# If you have a strong NVIDIA GPU, switch tree_method to "gpu_hist"
# model = xgb.XGBRegressor(tree_method="gpu_hist", predictor="gpu_predictor")

# Train
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Evaluate
rmse = root_mean_squared_error(y_test, y_pred)
print("rmse:", rmse)
r2 = r2_score(y_test, y_pred)
print("R²:", r2)

rmse: 0.04083974368377216
R²: 0.762247131939599


In [ ]:
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import mean_squared_error, r2_score
# from sklearn.neural_network import MLPRegressor
# from sklearn.preprocessing import StandardScaler

# # Features (all columns except target)
# X = df_rf.drop(columns=['current_speed_m_s'])
# y = df_rf['current_speed_m_s']

# # Train/test split
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.20, random_state=24
# )

# # Feature scaling (critical for MLPs)
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)

# # Define model with tuned parameters
# model = MLPRegressor(
#     hidden_layer_sizes=(128, 64, 32),  # deeper, tapered architecture
#     activation='tanh',                 # alternative to relu
#     solver='adam',                     # good for large datasets
#     learning_rate_init=0.01,          # initial learning rate
#     alpha=0.001,                       # L2 regularization
#     max_iter=2000,                     # more iterations for convergence
#     random_state=24
# )

# # Train
# model.fit(X_train_scaled, y_train)

# # Predict
# y_pred = model.predict(X_test_scaled)

# # Evaluate
# rmse = root_mean_squared_error(y_test, y_pred)
# print("rmse:", rmse)

# r2 = r2_score(y_test, y_pred)
# print("R²:", r2)

rmse: 0.06563114069001584
R²: 0.38598404799948893


In [ ]:
break

# pred

In [ ]:
df2 = pd.read_csv('target.csv', encoding='ISO-8859-1')

In [ ]:
target = df2[['date',"Temp. de l'eau", 'Salinité', 'Pression', 'Densité',
             'Période de vague','Hauteur de vague moy.', 'Hauteur de vague max.', 'Vitesse du vent',
            'Rafales', 'Direction du vent', 'Temp. air', 'Humidité','Courant-Direction(0m)',
            # 'Fluorescence','CDOM','Eau - Rétrodiffusion','Courant - Vitesse (6m)', 'Courant-Direction(6m)'
            ]]
print('len', len(target))
target.isna().sum()

len 2952


date                     0
Temp. de l'eau           0
Salinité                 0
Pression                 0
Densité                  0
Période de vague         0
Hauteur de vague moy.    0
Hauteur de vague max.    0
Vitesse du vent          0
Rafales                  0
Direction du vent        0
Temp. air                0
Humidité                 0
Courant-Direction(0m)    0
dtype: int64

In [ ]:
model.feature_names_in_

array(["Temp. de l'eau", 'Salinité', 'Pression', 'Densité',
       'Période de vague', 'Hauteur de vague moy.',
       'Hauteur de vague max.', 'Vitesse du vent', 'Rafales',
       'Direction du vent', 'Temp. air', 'Humidité',
       'Courant-Direction(0m)'], dtype='<U21')

In [ ]:
target['current_speed_m_s'] = model.predict(target.drop(columns='date'))
print(target.isna().sum())
target.head()

date                     0
Temp. de l'eau           0
Salinité                 0
Pression                 0
Densité                  0
Période de vague         0
Hauteur de vague moy.    0
Hauteur de vague max.    0
Vitesse du vent          0
Rafales                  0
Direction du vent        0
Temp. air                0
Humidité                 0
Courant-Direction(0m)    0
current_speed_m_s        0
dtype: int64


C:\Users\juvad3723\AppData\Local\Temp\ipykernel_25976\1771282251.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  target['current_speed_m_s'] = model.predict(target.drop(columns='date'))


,date,Temp. de l'eau,Salinité,Pression,Densité,Période de vague,Hauteur de vague moy.,Hauteur de vague max.,Vitesse du vent,Rafales,Direction du vent,Temp. air,Humidité,Courant-Direction(0m),current_speed_m_s
0,2025-07-25T14:00Z[UTC],15.01,30.50,100.80,1022.50,5.5,1.4,2.2,16.0,22.0,239.0,15.7,88.0,192.0,0.259608
1,2025-07-25T14:30Z[UTC],15.07,30.50,100.80,1022.49,5.4,1.4,2.3,13.0,19.0,246.0,15.9,88.0,212.0,0.257751
2,2025-07-25T15:00Z[UTC],15.12,30.55,100.79,1022.52,5.6,1.4,2.1,18.0,25.0,232.0,16.1,88.0,222.0,0.264130
3,2025-07-25T15:30Z[UTC],15.19,30.60,100.79,1022.55,5.5,1.2,1.7,16.0,27.0,245.0,16.1,88.0,262.0,0.296037
4,2025-07-25T16:00Z[UTC],15.23,30.66,100.79,1022.59,5.2,1.2,1.9,17.0,21.0,237.0,16.2,89.0,293.0,0.309982


In [ ]:
target.isna().any()

date                     False
Temp. de l'eau           False
Salinité                 False
Pression                 False
Densité                  False
Période de vague         False
Hauteur de vague moy.    False
Hauteur de vague max.    False
Vitesse du vent          False
Rafales                  False
Direction du vent        False
Temp. air                False
Humidité                 False
Courant-Direction(0m)    False
current_speed_m_s        False
dtype: bool

In [ ]:
y = df_rf['current_speed_m_s']

print('mean',y.mean().round(4))
print('median',y.median().round(4))
print('std',y.std().round(4))
print('min',y.min().round(4))
print('max',y.max().round(4))


b = target['current_speed_m_s']

print('mean',b.mean().round(4))
print('median',b.median().round(4))
print('std',b.std().round(4))
print('min',b.min().round(4))
print('max',b.max().round(4))

mean 0.2593
median 0.3
std 0.0839
min 0.0
max 0.9
mean 0.2424
median 0.2393
std 0.0334
min 0.1477
max 0.3516


In [ ]:
target.to_csv('2025_preds.csv')